In [1]:
%matplotlib widget
from re import I

from dvrptw_bench.dynamic.simulator import DynamicSimulator
from dvrptw_bench.rl.rl4co_policy import RL4COPolicy
from dvrptw_bench.rl.rl_model import build_attention_model
from dvrptw_bench.viz.inspector import inspect_dynamic
from dvrptw_bench.heuristics.ortools_dynamic import ORToolsDVRPTWSolver

from pathlib import Path

from pyarrow import dataset

from dvrptw_bench.data.instance_filters import find_rc_instances
from dvrptw_bench.data.solomon_parser import parse_solomon

dataset_path = Path("../../dataset/solomon_rc100")
instances = [parse_solomon(instance, max_customers=25, distance_scale=100) for instance in find_rc_instances(dataset_path)]
dod = 0.4
cutoff = 0.99
budget_s = 0.1
end_time_closeness = 0.9
soft_time_windows = True
# dynamic_instances = [build_dynamic_scenario(instance, epsilon=dod, seed=42) for instance in instances]
def solve_function(am, instance, time_limit_s, warm_start = None):
    policy = RL4COPolicy(am)
    warm_start =policy.infer_instance(instance)
    solver = ORToolsDVRPTWSolver(soft_time_windows=soft_time_windows)
    return solver.solve(instance, time_limit_s, warm_start)

model_weights = [f for f in Path("./model_weights/").glob("*.ckpt")][:1]
print("Model weights found:", model_weights)
sim = DynamicSimulator(instances[0])
sols = {}
for model_weight in model_weights:
    am = build_attention_model()
    am.load(model_weight)
    res = sim.run(lambda instance, time_limit_s, warm_start: solve_function(am, instance, time_limit_s, warm_start), budget_s=budget_s, epsilon=dod, seed=15, cutoff_ratio=cutoff)
    sols[model_weight.stem] = res[0]


/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'env' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['env'])`.
/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'policy' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['policy'])`.
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of t

Model weights found: [PosixPath('model_weights/attention-model-penalized-RC1-customers-25-penalty-0-lr-0.0005-seed-7960100.ckpt')]
CUstomers length: 25, n_dyn: 10, dynamic_ids: [2, 5, 8, 11, 12, 13, 15, 16, 22, 25]


/Users/giuseppe/Documents/personal/fyp-vrp/dvrptw-benchmark/.venv/lib/python3.12/site-packages/lightning/pytorch/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['baseline.baseline.policy.encoder.init_embedding.init_embed.weight', 'baseline.baseline.policy.encoder.init_embedding.init_embed.bias', 'baseline.baseline.policy.encoder.init_embedding.init_embed_depot.weight', 'baseline.baseline.policy.encoder.init_embedding.init_embed_depot.bias', 'baseline.baseline.policy.encoder.net.layers.0.0.module.Wqkv.weight', 'baseline.baseline.policy.encoder.net.layers.0.0.module.Wqkv.bias', 'baseline.baseline.policy.encoder.net.layers.0.0.module.out_proj.weight', 'baseline.baseline.policy.encoder.net.layers.0.0.module.out_proj.bias', 'baseline.baseline.policy.encoder.net.layers.0.1.normalizer.weight', 'baseline.baseline.policy.encoder.net.layers.0.1.normalizer.bias', 'baseline.baseline.policy.encoder.net.layers.0.1.normalizer.running_mean', 'baseline.baseli

: 